# Phase 1 — Legal Encoder-Only Language Model (Bidirectional GQA + MLM)

This notebook builds a **separate encoder-only model** using the same 50k A-1 BPE tokenizer, token shards, source mixing, and 512-token sequence setup as `phase1_gqa`. It does not modify the GQA or SWA checkpoints.

**Objective:** masked language modeling (MLM). Unlike the decoder-only GQA, each token can attend to both left and right context; about 15% of eligible input tokens are corrupted and the model predicts their original token IDs. The best checkpoint is selected by validation MLM loss and saved for later Phase 3 fine-tuning (NER/classification).

Run this notebook from the same project folder that contains `token_shards_a1_50k/` and `tokenizers_a1/legal_bpe_a1_50k_final.json`. Existing shards/indexes are reused; the expensive tokenizer training and corpus tokenization are not repeated.

## 1. Imports, paths, and runtime

In [1]:
import os
import json
import math
import time
import random
import gc
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torch.utils.checkpoint import checkpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.float16

# Use the same local artifacts as phase1_gqa.
TOKENIZER_PATH = Path("tokenizers_a1/legal_bpe_a1_50k_final.json")
SHARD_ROOT = Path("token_shards_a1_50k")
MANIFEST_PATH = SHARD_ROOT / "manifest.json"
CHECKPOINT_DIR = Path("phase1_encoder_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_encoder_mlm.pt"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_encoder_resume.pt"

CONTEXT_LENGTH = 512  # same as the GQA training context
VOCAB_SIZE = 50_000

print("torch:", torch.__version__)
print("device:", device)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu:", props.name)
    print(f"gpu memory: {props.total_memory / (1024**3):.2f} GiB")
print("tokenizer:", TOKENIZER_PATH.resolve())
print("shards:", SHARD_ROOT.resolve())
print("checkpoint directory:", CHECKPOINT_DIR.resolve())

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(f"Tokenizer file not found: {TOKENIZER_PATH}. Run from the project root or update TOKENIZER_PATH.")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Shard manifest not found: {MANIFEST_PATH}. Generate/reuse the Phase 1 token shards first.")

torch: 2.14.0+cu130
device: cuda
cuda available: True
gpu: NVIDIA GeForce RTX 5090
gpu memory: 31.35 GiB
tokenizer: /home/ugp/Downloads/tokenizers_a1/legal_bpe_a1_50k_final.json
shards: /home/ugp/Downloads/token_shards_a1_50k
checkpoint directory: /home/ugp/Downloads/phase1_encoder_checkpoints


## 2. Verify the exact Phase 1 tokenizer and load the existing shard manifest

In [2]:
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_payload = json.load(f)

itos = tokenizer_payload["itos"]
expected_special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
if len(itos) != VOCAB_SIZE:
    raise ValueError(f"Expected tokenizer vocab {VOCAB_SIZE:,}, found {len(itos):,}")
if list(itos[:4]) != expected_special_tokens:
    raise ValueError(f"Unexpected special-token order: {itos[:4]}")

PAD_TOKEN_ID, UNK_TOKEN_ID, BOS_TOKEN_ID, EOS_TOKEN_ID = range(4)
SPECIAL_TOKEN_IDS = {PAD_TOKEN_ID, UNK_TOKEN_ID, BOS_TOKEN_ID, EOS_TOKEN_ID}
# The frozen A-1 tokenizer has no dedicated [MASK] token. Keep its 50k vocabulary
# compatible with Phase 3 by using the existing <unk> ID as the corruption sentinel.
MASK_TOKEN_ID = UNK_TOKEN_ID

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

# Resolve manifest paths for the current machine while keeping manifest order unchanged,
# because sequence-index files refer to manifest row IDs.
for entry in manifest:
    shard_path = Path(entry["path"])
    if not shard_path.exists():
        candidates = [Path.cwd() / shard_path]
        if shard_path.parts and shard_path.parts[0] == SHARD_ROOT.name:
            candidates.append(Path.cwd().joinpath(*shard_path.parts[1:]))
        found = next((candidate for candidate in candidates if candidate.exists()), None)
        if found is None:
            raise FileNotFoundError(f"Shard from manifest was not found: {entry['path']}")
        shard_path = found
    entry["path"] = str(shard_path.resolve())

print("tokenizer vocab:", len(itos))
print("special tokens:", itos[:4])
print("manifest rows:", len(manifest))
print("first shard:", manifest[0]["path"] if manifest else "none")
print("mask sentinel ID:", MASK_TOKEN_ID, "(existing <unk>; tokenizer remains unchanged)")

assert CONTEXT_LENGTH == 512, "This notebook expects the Phase 1 GQA 512-token sequence index files."

tokenizer vocab: 50000
special tokens: ['<pad>', '<unk>', '<s>', '</s>']
manifest rows: 217
first shard: /home/ugp/Downloads/token_shards_a1_50k/train/acts/shard_00000.bin
mask sentinel ID: 1 (existing <unk>; tokenizer remains unchanged)


## 3. Reuse the GQA sequence indexes and legal shard dataset

In [3]:
source_names = ["acts", "high_courts", "supreme_court", "tribunals"]

train_indices = {
    source: np.load(SHARD_ROOT / f"train_{source}_sequence_index.npy")
    for source in source_names
}
val_indices = {
    source: np.load(SHARD_ROOT / f"val_{source}_sequence_index.npy")
    for source in source_names
}

class LegalShardDataset(Dataset):
    """Read fixed-length token windows lazily from the same uint16 shards as GQA."""
    def __init__(self, index, manifest_rows, context_length=CONTEXT_LENGTH):
        self.index = index
        self.manifest = manifest_rows
        self.context_length = context_length
        self.memmaps = {}

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        manifest_id, start = self.index[i]
        manifest_id = int(manifest_id)
        start = int(start)
        if manifest_id not in self.memmaps:
            self.memmaps[manifest_id] = np.memmap(
                self.manifest[manifest_id]["path"], dtype=np.uint16, mode="r"
            )
        data = self.memmaps[manifest_id]
        x = np.asarray(data[start:start + self.context_length], dtype=np.int64)
        y = np.asarray(data[start + 1:start + self.context_length + 1], dtype=np.int64)
        return torch.from_numpy(x.copy()), torch.from_numpy(y.copy())

train_datasets = {
    source: LegalShardDataset(train_indices[source], manifest, CONTEXT_LENGTH)
    for source in source_names
}
val_datasets = {
    source: LegalShardDataset(val_indices[source], manifest, CONTEXT_LENGTH)
    for source in source_names
}
val_dataset = ConcatDataset([val_datasets[source] for source in source_names])

for source in source_names:
    if len(train_datasets[source]) == 0 or len(val_datasets[source]) == 0:
        raise ValueError(f"Empty train/validation sequence index for source: {source}")
    x_sample, y_sample = train_datasets[source][0]
    assert x_sample.numel() == CONTEXT_LENGTH and y_sample.numel() == CONTEXT_LENGTH
    assert torch.equal(x_sample[1:], y_sample[:-1])
    print(f"{source:14s} train={len(train_datasets[source]):,} | val={len(val_datasets[source]):,}")

print("combined validation sequences:", f"{len(val_dataset):,}")

acts           train=240,437 | val=4,678
high_courts    train=2,415,688 | val=48,449
supreme_court  train=418,865 | val=9,443
tribunals      train=2,042,951 | val=40,973
combined validation sequences: 103,543


## 4. Keep the same source-mixing policy

In [4]:
class MixedLegalDataset(Dataset):
    """Same natural / smoothed / equal-downsampled mixing policy as phase1_gqa."""
    def __init__(self, train_datasets, mixing_strategy="smoothed", alpha=0.5, seed=42):
        self.train_datasets = train_datasets
        self.sources = list(train_datasets.keys())
        self.alpha = alpha
        self.rng = np.random.default_rng(seed)
        self.indices = {
            source: np.arange(len(train_datasets[source]), dtype=np.uint32)
            for source in self.sources
        }
        self.source_ids = {source: i for i, source in enumerate(self.sources)}
        self.mixing_strategy = mixing_strategy
        self.sequence_plan = self._build_plan()

    def _build_plan(self):
        lengths = np.array([len(self.indices[s]) for s in self.sources], dtype=np.float64)
        if self.mixing_strategy == "natural":
            counts = lengths.astype(np.int64)
        elif self.mixing_strategy == "equal_downsample":
            counts = np.full(len(self.sources), int(lengths.min()), dtype=np.int64)
        elif self.mixing_strategy == "smoothed":
            weights = lengths ** self.alpha
            probabilities = weights / weights.sum()
            max_total = int(np.floor(np.min(lengths / probabilities)))
            counts = np.floor(probabilities * max_total).astype(np.int64)
        else:
            raise ValueError("mixing_strategy must be natural, smoothed, or equal_downsample")

        plan_parts = []
        for source, count in zip(self.sources, counts):
            count = int(count)
            selected = self.rng.choice(self.indices[source], size=count, replace=False)
            source_id = self.source_ids[source]
            source_part = np.column_stack((np.full(count, source_id, dtype=np.uint32), selected))
            plan_parts.append(source_part)
        plan = np.concatenate(plan_parts, axis=0)
        self.rng.shuffle(plan)
        self.target_counts = {s: int(c) for s, c in zip(self.sources, counts)}
        self.source_probabilities = {s: float(c / len(plan)) for s, c in zip(self.sources, counts)}
        return plan

    def __len__(self):
        return len(self.sequence_plan)

    def __getitem__(self, i):
        source_id = int(self.sequence_plan[i, 0])
        local_index = int(self.sequence_plan[i, 1])
        source = self.sources[source_id]
        return self.train_datasets[source][local_index]

MIXING_STRATEGY = "smoothed"
MIXING_ALPHA = 0.5
print("mixing strategy:", MIXING_STRATEGY)
print("alpha:", MIXING_ALPHA)

mixing strategy: smoothed
alpha: 0.5


## 5. Encoder configuration — match the GQA parameter scale

In [5]:
@dataclass
class ModelConfig:
    vocab_size: int = VOCAB_SIZE
    d_model: int = 768
    n_layers: int = 16
    n_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = CONTEXT_LENGTH
    rope_base: float = 10_000.0
    rms_norm_eps: float = 1e-6
    dropout: float = 0.0

cfg = ModelConfig()
assert cfg.d_model % cfg.n_heads == 0
assert cfg.n_heads % cfg.n_kv_heads == 0
assert (cfg.d_model // cfg.n_heads) % 2 == 0
head_dim = cfg.d_model // cfg.n_heads
n_rep = cfg.n_heads // cfg.n_kv_heads

print(cfg)
print("head dimension:", head_dim)
print("KV repetition factor:", n_rep)
print("Attention mode: bidirectional (no causal mask)")
print("Pretraining objective: masked language modeling")

ModelConfig(vocab_size=50000, d_model=768, n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, max_seq_len=512, rope_base=10000.0, rms_norm_eps=1e-06, dropout=0.0)
head dimension: 64
KV repetition factor: 3
Attention mode: bidirectional (no causal mask)
Pretraining objective: masked language modeling


In [6]:
def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    assert d_head % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inv_freq)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    sin = sin.to(device=x.device, dtype=x.dtype).unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        input_dtype = x.dtype
        xf = x.float()
        norm = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (xf * norm * self.weight.float()).to(input_dtype)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

## 6. Bidirectional GQA attention and encoder blocks

In [7]:
class BidirectionalGQAAttention(nn.Module):
    """Grouped-query self-attention with RoPE and no causal/local attention mask."""
    def __init__(self, config):
        super().__init__()
        self.d_model = config.d_model
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = config.n_heads // config.n_kv_heads
        self.max_seq_len = config.max_seq_len
        self.dropout = config.dropout

        self.q_proj = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=False)

        cos, sin = build_rope_cache(config.max_seq_len, self.head_dim, config.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _split_heads(self, x, n_heads):
        batch_size, seq_len, _ = x.shape
        return x.reshape(batch_size, seq_len, n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        if seq_len > self.max_seq_len:
            raise ValueError(f"sequence length {seq_len} exceeds {self.max_seq_len}")

        q = self._split_heads(self.q_proj(x), self.n_heads)
        k = self._split_heads(self.k_proj(x), self.n_kv_heads)
        v = self._split_heads(self.v_proj(x), self.n_kv_heads)

        cos = self.rope_cos[:seq_len]
        sin = self.rope_sin[:seq_len]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # Expand compact KV heads to query-head count. SDPA may use optimized kernels
        # when supported by the installed PyTorch/GPU combination.
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        attn = attn.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(attn)


class EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.attn = BidirectionalGQAAttention(config)
        self.ffn_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.mlp = SwiGLU(config.d_model, config.d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        x = x + self.mlp(self.ffn_norm(x))
        return x

## 7. Full encoder and tied masked-token prediction head

In [8]:
class LegalEncoderForMLM(nn.Module):
    def __init__(self, config, gradient_checkpointing=True):
        super().__init__()
        self.config = config
        self.gradient_checkpointing = gradient_checkpointing
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList([EncoderBlock(config) for _ in range(config.n_layers)])
        self.final_norm = RMSNorm(config.d_model, config.rms_norm_eps)
        self.mlm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.apply(self._init_weights)
        # Weight tying keeps the output head parameter-free beyond the input embedding.
        self.mlm_head.weight = self.token_embedding.weight
        residual_std = 0.02 / math.sqrt(2 * config.n_layers)
        for layer in self.layers:
            nn.init.normal_(layer.attn.out_proj.weight, mean=0.0, std=residual_std)
            nn.init.normal_(layer.mlp.down_proj.weight, mean=0.0, std=residual_std)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode(self, input_ids):
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            if self.gradient_checkpointing and self.training:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.final_norm(x)

    def forward(self, input_ids, return_hidden=False):
        hidden = self.encode(input_ids)
        logits = F.linear(hidden, self.token_embedding.weight)
        if return_hidden:
            return logits, hidden
        return logits

head_dim = cfg.d_model // cfg.n_heads
vocab_params = cfg.vocab_size * cfg.d_model
attention_params = cfg.d_model * (cfg.n_heads * head_dim + 2 * cfg.n_kv_heads * head_dim + cfg.d_model)
swiglu_params = 3 * cfg.d_model * cfg.d_ff
norm_params_per_layer = 2 * cfg.d_model
params_per_layer = attention_params + swiglu_params + norm_params_per_layer
num_params = vocab_params + cfg.n_layers * params_per_layer + cfg.d_model
print(f"estimated trainable parameters: {num_params:,} ({num_params / 1e6:.2f}M)")
print("MLM head parameters added separately: 0 (tied to token embeddings)")
assert 130_000_000 <= num_params <= 150_000_000, "Unexpected parameter count; verify ModelConfig."

estimated trainable parameters: 139,088,640 (139.09M)
MLM head parameters added separately: 0 (tied to token embeddings)


## 8. Architecture smoke test

In [9]:
# Use a small temporary configuration to catch shape / mask errors without allocating a second 140M model.
small_cfg = ModelConfig(vocab_size=VOCAB_SIZE, d_model=96, n_layers=2, n_heads=4,
                        n_kv_heads=2, d_ff=256, max_seq_len=32)
small_model = LegalEncoderForMLM(small_cfg, gradient_checkpointing=False).to(device)
small_ids = torch.randint(4, VOCAB_SIZE, (2, 16), device=device)
small_logits, small_hidden = small_model(small_ids, return_hidden=True)
assert small_logits.shape == (2, 16, VOCAB_SIZE)
assert small_hidden.shape == (2, 16, 96)
print("logits shape:", tuple(small_logits.shape))
print("hidden shape:", tuple(small_hidden.shape))
del small_model, small_ids, small_logits, small_hidden
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

logits shape: (2, 16, 50000)
hidden shape: (2, 16, 96)


## 9. Training configuration (same token budget; memory-aware micro-batch)

In [10]:
# Preserve the GQA effective batch target: 64 sequences x 512 tokens = 32,768 tokens/update.
TARGET_EFFECTIVE_SEQUENCES = 64
if device.type != "cuda":
    MICRO_BATCH_SIZE = 1
else:
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if gpu_mem_gb <= 6:
        MICRO_BATCH_SIZE = 1
    elif gpu_mem_gb <= 10:
        MICRO_BATCH_SIZE = 2
    elif gpu_mem_gb <= 20:
        MICRO_BATCH_SIZE = 4
    else:
        MICRO_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = max(1, TARGET_EFFECTIVE_SEQUENCES // MICRO_BATCH_SIZE)
EFFECTIVE_SEQUENCES_PER_UPDATE = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
EFFECTIVE_TOKENS_PER_UPDATE = EFFECTIVE_SEQUENCES_PER_UPDATE * CONTEXT_LENGTH

MAX_TRAIN_TOKENS = 500_000_000
MAX_STEPS = math.ceil(MAX_TRAIN_TOKENS / EFFECTIVE_TOKENS_PER_UPDATE)
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.1
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.95
ADAM_EPS = 1e-8
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.02
WARMUP_STEPS = max(1, int(MAX_STEPS * WARMUP_RATIO))
VALIDATE_EVERY_STEPS = 1_000
VALIDATION_TOKENS = 10_000_000
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 0.001
CHECKPOINT_EVERY_STEPS = 1_000
CHECKPOINT_EVERY_MINUTES = 15
NUM_WORKERS = 0  # safest for local Windows/Jupyter + mmap datasets
PREFETCH_FACTOR = 2
MLM_PROBABILITY = 0.15
VALIDATION_MASK_SEED = 12345

print("micro-batch:", MICRO_BATCH_SIZE)
print("gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("effective tokens/update:", f"{EFFECTIVE_TOKENS_PER_UPDATE:,}")
print("maximum updates:", f"{MAX_STEPS:,}")
print("learning rate:", LEARNING_RATE)
print("validation interval:", VALIDATE_EVERY_STEPS)
print("validation MLM tokens (approx):", f"{VALIDATION_TOKENS:,}")
print("AMP enabled:", USE_AMP)


def lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return (current_step + 1) / WARMUP_STEPS
    progress = (current_step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

micro-batch: 8
gradient accumulation: 8
effective tokens/update: 32,768
maximum updates: 15,259
learning rate: 0.0002
validation interval: 1000
validation MLM tokens (approx): 10,000,000
AMP enabled: True


## 10. MLM corruption: make input tokens and prediction labels

In [11]:
def make_mlm_batch(input_ids, generator=None):
    """Create BERT-style 15% MLM corruption while preserving the fixed 50k tokenizer."""
    original = input_ids
    corrupted = original.clone()
    labels = torch.full_like(original, -100)

    eligible = torch.ones_like(original, dtype=torch.bool)
    for special_id in SPECIAL_TOKEN_IDS:
        eligible &= original.ne(special_id)

    selected = (torch.rand(original.shape, device=original.device, generator=generator) < MLM_PROBABILITY) & eligible

    # Guarantee at least one prediction target per row when the row has any ordinary token.
    for row in range(original.shape[0]):
        if not selected[row].any():
            candidates = torch.nonzero(eligible[row], as_tuple=False).flatten()
            if candidates.numel() > 0:
                pick = torch.randint(candidates.numel(), (1,), device=original.device, generator=generator)
                selected[row, candidates[pick]] = True

    labels[selected] = original[selected]
    replace_draw = torch.rand(original.shape, device=original.device, generator=generator)
    mask_positions = selected & (replace_draw < 0.80)
    random_positions = selected & (replace_draw >= 0.80) & (replace_draw < 0.90)
    # The final 10% of selected tokens are intentionally left unchanged.
    corrupted[mask_positions] = MASK_TOKEN_ID

    random_ids = torch.randint(4, VOCAB_SIZE, original.shape, device=original.device, generator=generator)
    corrupted[random_positions] = random_ids[random_positions]
    return corrupted, labels


# Quick check: only selected positions contribute to the loss.
probe = torch.tensor([[2, 101, 102, 3, 103, 104, 105, 106]], dtype=torch.long, device=device)
probe_masked, probe_labels = make_mlm_batch(probe)
print("original: ", probe.tolist())
print("corrupted:", probe_masked.tolist())
print("labels:   ", probe_labels.tolist())
print("masked targets:", int((probe_labels != -100).sum()))
assert torch.all(probe_labels[probe_labels != -100] == probe[probe_labels != -100])
del probe, probe_masked, probe_labels

original:  [[2, 101, 102, 3, 103, 104, 105, 106]]
corrupted: [[2, 1, 102, 3, 103, 104, 105, 106]]
labels:    [[-100, 101, -100, -100, -100, -100, -100, -100]]
masked targets: 1


## 11. Fixed validation subset and masked-LM validation loss

In [12]:
VALIDATION_SEQUENCES = min(len(val_dataset), max(1, VALIDATION_TOKENS // CONTEXT_LENGTH))
validation_rng = np.random.default_rng(VALIDATION_MASK_SEED)
validation_indices = np.sort(validation_rng.choice(len(val_dataset), size=VALIDATION_SEQUENCES, replace=False))
val_eval_dataset = Subset(val_dataset, validation_indices.tolist())
val_eval_loader = DataLoader(
    val_eval_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
print("validation sequences:", f"{VALIDATION_SEQUENCES:,}")
print("approx validation tokens:", f"{VALIDATION_SEQUENCES * CONTEXT_LENGTH:,}")


def evaluate_model(eval_model, loader):
    eval_model.eval()
    # Resetting the generator makes the validation corruptions identical at each evaluation.
    val_generator = torch.Generator(device=device)
    val_generator.manual_seed(VALIDATION_MASK_SEED)
    total_loss = 0.0
    total_masked = 0
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device, non_blocking=True)
            masked_x, labels = make_mlm_batch(x, generator=val_generator)
            with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = eval_model(masked_x)
                loss_sum = F.cross_entropy(
                    logits.reshape(-1, cfg.vocab_size),
                    labels.reshape(-1),
                    ignore_index=-100,
                    reduction="sum",
                )
            n_masked = int((labels != -100).sum().item())
            total_loss += float(loss_sum.float().item())
            total_masked += n_masked
    eval_model.train()
    return total_loss / max(total_masked, 1), total_masked

# The full validation pass runs at the first configured validation step (step 1000).
# The earlier small-model and MLM-corruption smoke tests check shapes/data without creating
# a second full-sized 140M-parameter model.

validation sequences: 19,531
approx validation tokens: 9,999,872


## 12. Optimizer, scheduler, and AMP scaler

In [13]:
model = LegalEncoderForMLM(cfg, gradient_checkpointing=True).to(device)
model.train()

decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.ndim == 1:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=(ADAM_BETA1, ADAM_BETA2),
    eps=ADAM_EPS,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=True) if USE_AMP else None
print("model and optimizer ready")
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("gradient checkpointing:", model.gradient_checkpointing)
print("AMP scaler:", scaler is not None)

model and optimizer ready
parameters: 139,088,640
gradient checkpointing: True
AMP scaler: True


## 13. Deterministic cycle-aware data stream

In [14]:
def build_train_loader(cycle_index):
    cycle_dataset = MixedLegalDataset(
        train_datasets,
        mixing_strategy=MIXING_STRATEGY,
        alpha=MIXING_ALPHA,
        seed=SEED + int(cycle_index),
    )
    loader = DataLoader(
        cycle_dataset,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=False,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return cycle_dataset, loader


def training_batch_stream(start_cycle=0, start_micro_batch=0):
    cycle_index = int(start_cycle)
    micro_batch_offset = int(start_micro_batch)
    while True:
        cycle_dataset, cycle_loader = build_train_loader(cycle_index)
        cycle_batches = len(cycle_loader)
        if micro_batch_offset >= cycle_batches:
            cycle_index += 1
            micro_batch_offset = 0
            del cycle_loader, cycle_dataset
            gc.collect()
            continue
        for batch_index, batch in enumerate(cycle_loader):
            if batch_index < micro_batch_offset:
                continue
            next_cycle = cycle_index
            next_offset = batch_index + 1
            if next_offset >= cycle_batches:
                next_cycle = cycle_index + 1
                next_offset = 0
            yield batch, next_cycle, next_offset
        del cycle_loader, cycle_dataset
        gc.collect()
        cycle_index += 1
        micro_batch_offset = 0

## 14. Best-checkpoint and resume-checkpoint helpers

In [15]:
def get_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def set_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda") is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def atomic_torch_save(payload, path):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temp_path)
    os.replace(temp_path, path)


def save_best_checkpoint(path, model, step, tokens_seen, best_val_loss):
    payload = {
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "model_config": asdict(cfg),
        "global_step": int(step),
        "tokens_seen": int(tokens_seen),
        "best_val_mlm_loss": float(best_val_loss),
        "pretraining_objective": "masked_language_modeling",
        "architecture": "bidirectional_gqa_encoder",
        "mask_token_id": int(MASK_TOKEN_ID),
        "tokenizer_path": str(TOKENIZER_PATH),
        "context_length": int(CONTEXT_LENGTH),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    atomic_torch_save(payload, path)


def save_resume_checkpoint(path, model, optimizer, scheduler, scaler,
                           global_step, tokens_seen, cycle_index, micro_batch_offset,
                           best_val_loss, early_stop_counter):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "global_step": int(global_step),
        "tokens_seen": int(tokens_seen),
        "cycle_index": int(cycle_index),
        "micro_batch_offset": int(micro_batch_offset),
        "best_val_mlm_loss": float(best_val_loss),
        "early_stop_counter": int(early_stop_counter),
        "rng_state": get_rng_state(),
        "model_config": asdict(cfg),
        "training_config": {
            "micro_batch_size": int(MICRO_BATCH_SIZE),
            "gradient_accumulation_steps": int(GRADIENT_ACCUMULATION_STEPS),
            "context_length": int(CONTEXT_LENGTH),
            "effective_tokens_per_update": int(EFFECTIVE_TOKENS_PER_UPDATE),
            "max_train_tokens": int(MAX_TRAIN_TOKENS),
            "max_steps": int(MAX_STEPS),
            "learning_rate": float(LEARNING_RATE),
            "mask_probability": float(MLM_PROBABILITY),
        },
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    atomic_torch_save(payload, path)


def load_resume_checkpoint(path, model, optimizer, scheduler, scaler):
    checkpoint_data = torch.load(path, map_location=device, weights_only=False)
    if checkpoint_data["model_config"] != asdict(cfg):
        raise ValueError("Checkpoint ModelConfig differs from the current notebook configuration.")
    model.load_state_dict(checkpoint_data["model_state_dict"])
    optimizer.load_state_dict(checkpoint_data["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint_data["scheduler_state_dict"])
    if scaler is not None and checkpoint_data.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint_data["scaler_state_dict"])
    set_rng_state(checkpoint_data["rng_state"])
    return checkpoint_data

print("best checkpoint:", BEST_CHECKPOINT_PATH)
print("resume checkpoint:", LAST_CHECKPOINT_PATH)

best checkpoint: phase1_encoder_checkpoints/best_encoder_mlm.pt
resume checkpoint: phase1_encoder_checkpoints/last_encoder_resume.pt


## 15. Start or resume the full pretraining run

In [16]:
RESUME_TRAINING = False  # set True after an interruption if last_encoder_resume.pt exists

if RESUME_TRAINING:
    if not LAST_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Cannot resume; checkpoint missing: {LAST_CHECKPOINT_PATH}")
    resume_data = load_resume_checkpoint(LAST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler)
    global_step = int(resume_data["global_step"])
    tokens_seen = int(resume_data["tokens_seen"])
    cycle_index = int(resume_data["cycle_index"])
    micro_batch_offset = int(resume_data["micro_batch_offset"])
    best_val_loss = float(resume_data["best_val_mlm_loss"])
    early_stop_counter = int(resume_data["early_stop_counter"])
    print("resumed at step:", global_step)
else:
    # Avoid accidentally mistaking a checkpoint from a previous fresh run for this run's result.
    for stale_path in (BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH):
        if stale_path.exists():
            stale_path.unlink()
    global_step = 0
    tokens_seen = 0
    cycle_index = 0
    micro_batch_offset = 0
    best_val_loss = float("inf")
    early_stop_counter = 0
    print("starting a fresh encoder pretraining run")

batch_stream = training_batch_stream(cycle_index, micro_batch_offset)
model.train()
optimizer.zero_grad(set_to_none=True)
running_loss = 0.0
running_updates = 0
last_checkpoint_time = time.time()
log_window_start = time.time()
log_window_tokens = tokens_seen
skipped_updates = 0

print("=" * 72)
print("STARTING ENCODER-ONLY MLM PRETRAINING")
print("=" * 72)
print("step:", f"{global_step:,} / {MAX_STEPS:,}")
print("tokens seen:", f"{tokens_seen:,}")
print("context:", CONTEXT_LENGTH)
print("micro-batch x accumulation:", MICRO_BATCH_SIZE, "x", GRADIENT_ACCUMULATION_STEPS)
print("effective tokens/update:", f"{EFFECTIVE_TOKENS_PER_UPDATE:,}")
print("best checkpoint will be saved to:", BEST_CHECKPOINT_PATH)

try:
    while global_step < MAX_STEPS:
        optimizer.zero_grad(set_to_none=True)
        update_loss_sum = 0.0
        update_micro_batches = 0

        for _ in range(GRADIENT_ACCUMULATION_STEPS):
            (x, _), cycle_index, micro_batch_offset = next(batch_stream)
            x = x.to(device, non_blocking=True)
            masked_x, labels = make_mlm_batch(x)

            with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = model(masked_x)
                loss = F.cross_entropy(
                    logits.reshape(-1, cfg.vocab_size),
                    labels.reshape(-1),
                    ignore_index=-100,
                    reduction="mean",
                )
                scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

            if scaler is not None:
                scaler.scale(scaled_loss).backward()
            else:
                scaled_loss.backward()

            update_loss_sum += float(loss.detach().float().item())
            update_micro_batches += 1
            tokens_seen += int(x.numel())

        if scaler is not None:
            scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        skipped = False
        if scaler is not None:
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped = scaler.get_scale() < old_scale
        else:
            optimizer.step()

        optimizer.zero_grad(set_to_none=True)
        if skipped:
            skipped_updates += 1
        else:
            scheduler.step()
            global_step += 1

        update_loss = update_loss_sum / max(update_micro_batches, 1)
        running_loss += update_loss
        running_updates += 1

        if global_step > 0 and global_step % 100 == 0 and not skipped:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            now = time.time()
            elapsed = max(now - log_window_start, 1e-8)
            tokens_per_sec = (tokens_seen - log_window_tokens) / elapsed
            mean_loss = running_loss / max(running_updates, 1)
            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"step {global_step:>7,} | train MLM loss {mean_loss:.4f} | "
                f"lr {current_lr:.2e} | tokens/s {tokens_per_sec:,.0f} | "
                f"grad {float(grad_norm):.3f}"
            )
            running_loss = 0.0
            running_updates = 0
            log_window_start = now
            log_window_tokens = tokens_seen

        should_validate = global_step > 0 and global_step % VALIDATE_EVERY_STEPS == 0 and not skipped
        if should_validate:
            print("\n" + "-" * 72)
            print("VALIDATION AT STEP", f"{global_step:,}")
            val_start = time.time()
            val_mlm_loss, val_masked_count = evaluate_model(model, val_eval_loader)
            print(f"validation MLM loss: {val_mlm_loss:.6f} over {val_masked_count:,} masked tokens")
            print(f"validation duration: {time.time() - val_start:.1f} s")

            improved = val_mlm_loss < best_val_loss - EARLY_STOP_MIN_DELTA
            if improved:
                best_val_loss = val_mlm_loss
                early_stop_counter = 0
                save_best_checkpoint(BEST_CHECKPOINT_PATH, model, global_step, tokens_seen, best_val_loss)
                print("NEW BEST CHECKPOINT SAVED:", BEST_CHECKPOINT_PATH)
            else:
                early_stop_counter += 1
                print(f"no significant improvement: {early_stop_counter}/{EARLY_STOP_PATIENCE}")

            save_resume_checkpoint(
                LAST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
                global_step, tokens_seen, cycle_index, micro_batch_offset,
                best_val_loss, early_stop_counter,
            )
            last_checkpoint_time = time.time()
            print("resume checkpoint saved")
            print("-" * 72 + "\n")

            if early_stop_counter >= EARLY_STOP_PATIENCE:
                print("early stopping triggered")
                break

        checkpoint_by_steps = global_step > 0 and global_step % CHECKPOINT_EVERY_STEPS == 0 and not skipped
        checkpoint_by_time = (time.time() - last_checkpoint_time) >= CHECKPOINT_EVERY_MINUTES * 60
        if (checkpoint_by_steps or checkpoint_by_time) and not should_validate and not skipped:
            save_resume_checkpoint(
                LAST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
                global_step, tokens_seen, cycle_index, micro_batch_offset,
                best_val_loss, early_stop_counter,
            )
            last_checkpoint_time = time.time()
            print(f"resume checkpoint saved at step {global_step:,}")

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    save_resume_checkpoint(
        LAST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
        global_step, tokens_seen, cycle_index, micro_batch_offset,
        best_val_loss, early_stop_counter,
    )
    print("training loop ended")
    print("final step:", f"{global_step:,}")
    print("tokens seen:", f"{tokens_seen:,}")
    print("best validation MLM loss:", best_val_loss)
    print("skipped AMP updates:", skipped_updates)
    print("best checkpoint exists:", BEST_CHECKPOINT_PATH.exists())
    print("resume checkpoint exists:", LAST_CHECKPOINT_PATH.exists())

except KeyboardInterrupt:
    print("\nKeyboardInterrupt received; saving resumable checkpoint...")
    save_resume_checkpoint(
        LAST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler,
        global_step, tokens_seen, cycle_index, micro_batch_offset,
        best_val_loss, early_stop_counter,
    )
    print("saved:", LAST_CHECKPOINT_PATH)

finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

starting a fresh encoder pretraining run
STARTING ENCODER-ONLY MLM PRETRAINING
step: 0 / 15,259
tokens seen: 0
context: 512
micro-batch x accumulation: 8 x 8
effective tokens/update: 32,768
best checkpoint will be saved to: phase1_encoder_checkpoints/best_encoder_mlm.pt
step     100 | train MLM loss 5.7967 | lr 6.62e-05 | tokens/s 35,647 | grad 3.990
step     200 | train MLM loss 3.9111 | lr 1.32e-04 | tokens/s 35,942 | grad 2.975
step     300 | train MLM loss 3.3720 | lr 1.97e-04 | tokens/s 35,966 | grad 2.387
step     400 | train MLM loss 2.9999 | lr 2.00e-04 | tokens/s 36,289 | grad 1.471
step     500 | train MLM loss 2.7432 | lr 2.00e-04 | tokens/s 36,402 | grad 1.730
step     600 | train MLM loss 2.5554 | lr 2.00e-04 | tokens/s 36,089 | grad 1.572
step     700 | train MLM loss 2.4364 | lr 2.00e-04 | tokens/s 36,061 | grad 1.511
step     800 | train MLM loss 2.3357 | lr 1.99e-04 | tokens/s 36,013 | grad 1.322
step     900 | train MLM loss 2.2620 | lr 1.99e-04 | tokens/s 36,095 | gr

## 16. Reload and verify the best encoder checkpoint for Phase 3

In [17]:
if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"No best checkpoint found at {BEST_CHECKPOINT_PATH}. "
        "The model saves a best checkpoint after the first validation improvement; "
        "check that training reached a validation step."
    )

best_payload = torch.load(BEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
phase3_encoder = LegalEncoderForMLM(ModelConfig(**best_payload["model_config"]), gradient_checkpointing=False).to(device)
phase3_encoder.load_state_dict(best_payload["model_state_dict"])
phase3_encoder.eval()

print("best checkpoint:", BEST_CHECKPOINT_PATH.resolve())
print("best step:", best_payload["global_step"])
print("best validation MLM loss:", best_payload["best_val_mlm_loss"])
print("architecture:", best_payload["architecture"])
print("objective:", best_payload["pretraining_objective"])
print("tokenizer:", best_payload["tokenizer_path"])

# Downstream models can use this representation for token classification or sequence classification.
with torch.no_grad():
    demo_ids = torch.randint(4, VOCAB_SIZE, (1, 16), device=device)
    demo_hidden = phase3_encoder.encode(demo_ids)
    demo_logits = phase3_encoder(demo_ids)
print("encoder hidden states:", tuple(demo_hidden.shape))  # [batch, sequence, d_model]
print("MLM logits:", tuple(demo_logits.shape))              # [batch, sequence, vocab_size]

# For Phase 3, load BEST_CHECKPOINT_PATH and use `phase3_encoder.encode(input_ids)` as the backbone.
# Attach a token-classification head for NER or a pooled/CLS-style head for sequence classification.

best checkpoint: /home/ugp/Downloads/phase1_encoder_checkpoints/best_encoder_mlm.pt
best step: 15000
best validation MLM loss: 1.0438882003729564
architecture: bidirectional_gqa_encoder
objective: masked_language_modeling
tokenizer: tokenizers_a1/legal_bpe_a1_50k_final.json
encoder hidden states: (1, 16, 768)
MLM logits: (1, 16, 50000)


## What changed from GQA?

- **Attention:** GQA query/key/value projections are retained, but the causal mask and KV-cache path are removed. Every token attends bidirectionally to the full 512-token window.
- **Objective:** next-token cross-entropy is replaced by **15% masked language modeling**. Only selected token positions contribute to the loss; the validation metric is validation MLM loss (not autoregressive perplexity).
- **Output:** the input embedding and MLM prediction head share weights. `encode(input_ids)` returns contextual hidden states for NER/classification fine-tuning.
- **Kept the same:** 50k A-1 BPE tokenizer, `token_shards_a1_50k/`, manifest and sequence-index files, source names, smoothed source mixing (`alpha=0.5`), 512-token context, 768 hidden size, 16 layers, 12 query heads / 4 KV heads, SwiGLU, RMSNorm, RoPE, AdamW, cosine schedule, and the 500M-token target budget.
- **Checkpoint outputs:** `phase1_encoder_checkpoints/best_encoder_mlm.pt` (best validation model for Phase 3) and `last_encoder_resume.pt` (optimizer + data position for resuming interrupted training).

**Important tokenizer compatibility choice:** the existing tokenizer only defines `<pad>`, `<unk>`, `<s>`, and `</s>`. To avoid changing the Phase 1 tokenizer or vocabulary size, the notebook uses `<unk>` (ID 1) as the MLM corruption sentinel rather than adding a new `[MASK]` ID.